In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
import pickle
warnings.filterwarnings('ignore')

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier

# Pipeline
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

# Evaluation
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
    precision_recall_curve, average_precision_score,
    f1_score, accuracy_score
)
from sklearn.calibration import calibration_curve
from sklearn.model_selection import cross_val_score, StratifiedKFold

# Settings
os.chdir('/Users/cirrus/Desktop/PMOS_PROJECT/notebook')
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

print('✅ All libraries imported successfully')

✅ All libraries imported successfully


In [2]:
# Load train/test splits saved in Block 2
X_train = pd.read_csv('artifacts/X_train.csv')
X_test  = pd.read_csv('artifacts/X_test.csv')
y_train = pd.read_csv('artifacts/y_train.csv').squeeze()
y_test  = pd.read_csv('artifacts/y_test.csv').squeeze()

# Load scaled versions
X_train_scaled = pd.read_csv('artifacts/X_train_scaled.csv')
X_test_scaled  = pd.read_csv('artifacts/X_test_scaled.csv')

# Load feature list
with open('artifacts/final_features.pkl', 'rb') as f:
    final_features = pickle.load(f)

print('✅ All data loaded successfully')
print(f'   X_train : {X_train.shape}')
print(f'   X_test  : {X_test.shape}')
print(f'   y_train : {y_train.shape} | Classes: {y_train.value_counts().to_dict()}')
print(f'   y_test  : {y_test.shape}  | Classes: {y_test.value_counts().to_dict()}')
print(f'\n   Features: {final_features}')

✅ All data loaded successfully
   X_train : (550, 13)
   X_test  : (109, 13)
   y_train : (550,) | Classes: {0: 275, 1: 275}
   y_test  : (109,)  | Classes: {0: 73, 1: 36}

   Features: ['Follicle No. (R)', 'hair growth(Y/N)', 'Skin darkening (Y/N)', 'Weight gain(Y/N)', 'Cycle(R/I)', 'Follicle No. (L)', 'Pimples(Y/N)', 'LH(mIU/mL)', 'Fast food (Y/N)', 'AMH(ng/mL)', 'Marraige Status (Yrs)', 'I   beta-HCG(mIU/mL)', 'PRG(ng/mL)']


In [3]:
def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    # Predictions
    y_pred      = model.predict(X_te)
    y_prob      = model.predict_proba(X_te)[:, 1]
    
    # Metrics
    acc         = accuracy_score(y_te, y_pred)
    f1          = f1_score(y_te, y_pred)
    roc_auc     = roc_auc_score(y_te, y_prob)
    pr_auc      = average_precision_score(y_te, y_prob)
    
    # Cross validation
    cv          = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    cv_scores   = cross_val_score(model, X_tr, y_tr, cv=cv, scoring='roc_auc')
    
    print(f'\n{"="*45}')
    print(f'  {name}')
    print(f'{"="*45}')
    print(f'  Accuracy       : {acc:.4f}')
    print(f'  F1 Score       : {f1:.4f}')
    print(f'  ROC-AUC        : {roc_auc:.4f}')
    print(f'  PR-AUC         : {pr_auc:.4f}')
    print(f'  CV ROC-AUC     : {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
    print(f'\n  Classification Report:')
    print(classification_report(y_te, y_pred, 
          target_names=['PMOS-', 'PMOS+']))
    
    return {
        'Model'    : name,
        'Accuracy' : round(acc, 4),
        'F1'       : round(f1, 4),
        'ROC-AUC'  : round(roc_auc, 4),
        'PR-AUC'   : round(pr_auc, 4),
        'CV-AUC'   : round(cv_scores.mean(), 4),
        'CV-STD'   : round(cv_scores.std(), 4),
        'model_obj': model,
        'y_prob'   : y_prob
    }

print('✅ Evaluation function defined')

✅ Evaluation function defined


### Cell 4 — Logistic Regression (Baseline Model)

In [4]:
# Logistic Regression Pipeline
pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(random_state=42, max_iter=1000))
])

pipe_lr.fit(X_train, y_train)

results = []
lr_results = evaluate_model(
    'Logistic Regression',
    pipe_lr, X_train, y_train,
    X_test, y_test
)
results.append(lr_results)

# Save model
with open('artifacts/model_lr.pkl', 'wb') as f:
    pickle.dump(pipe_lr, f)
print('\n✅ Logistic Regression model saved')


  Logistic Regression
  Accuracy       : 0.9174
  F1 Score       : 0.8800
  ROC-AUC        : 0.9654
  PR-AUC         : 0.9617
  CV ROC-AUC     : 0.9556 ± 0.0219

  Classification Report:
              precision    recall  f1-score   support

       PMOS-       0.96      0.92      0.94        73
       PMOS+       0.85      0.92      0.88        36

    accuracy                           0.92       109
   macro avg       0.90      0.92      0.91       109
weighted avg       0.92      0.92      0.92       109


✅ Logistic Regression model saved


#### Cell 4 — Reasoning
- Linear baseline — if complex models can't beat this, they aren't worth the complexity.
- ROC-AUC = 0.9654 → ranks PMOS+ above PMOS- 96.5% of the time.
- PMOS+ Recall = 0.92 → misses only 8% of actual cases.
- In clinical terms: missing a PMOS diagnosis is far worse than a false alarm —
- high recall is the priority metric, not accuracy.
- CV-AUC = 0.9556 ± 0.022 → consistent across folds, model is stable.

### Cell 5 — Random Forest

In [5]:
pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        n_jobs=-1
    ))
])

pipe_rf.fit(X_train, y_train)

rf_results = evaluate_model(
    'Random Forest',
    pipe_rf, X_train, y_train,
    X_test, y_test
)
results.append(rf_results)

# Save model
with open('artifacts/model_rf.pkl', 'wb') as f:
    pickle.dump(pipe_rf, f)
print('\n✅ Random Forest model saved')


  Random Forest
  Accuracy       : 0.9174
  F1 Score       : 0.8732
  ROC-AUC        : 0.9585
  PR-AUC         : 0.9387
  CV ROC-AUC     : 0.9718 ± 0.0192

  Classification Report:
              precision    recall  f1-score   support

       PMOS-       0.93      0.95      0.94        73
       PMOS+       0.89      0.86      0.87        36

    accuracy                           0.92       109
   macro avg       0.91      0.90      0.91       109
weighted avg       0.92      0.92      0.92       109


✅ Random Forest model saved


#### Cell 5 — Random Forest Reasoning
- Ensemble of 100 decision trees — expected to beat LR on non-linear patterns.
- Surprisingly LR outperforms RF on test set — especially PMOS+ Recall (0.92 vs 0.86).
- RF misses 5 PMOS+ patients vs LR missing only 3 — clinically significant difference.
- CV-AUC = 0.972 — RF generalises well but LR remains stronger on this dataset.
- Signal in our 13 features is largely linear — explains why LR performs competitively.

### Cell 6 — XGBoost

In [6]:
pipe_xgb = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=4,
        random_state=42,
        eval_metric='logloss',
        verbosity=0
    ))
])

pipe_xgb.fit(X_train, y_train)

xgb_results = evaluate_model(
    'XGBoost',
    pipe_xgb, X_train, y_train,
    X_test, y_test
)
results.append(xgb_results)

# Save model
with open('artifacts/model_xgb.pkl', 'wb') as f:
    pickle.dump(pipe_xgb, f)
print('\n✅ XGBoost model saved')


  XGBoost
  Accuracy       : 0.9266
  F1 Score       : 0.8857
  ROC-AUC        : 0.9650
  PR-AUC         : 0.9448
  CV ROC-AUC     : 0.9714 ± 0.0164

  Classification Report:
              precision    recall  f1-score   support

       PMOS-       0.93      0.96      0.95        73
       PMOS+       0.91      0.86      0.89        36

    accuracy                           0.93       109
   macro avg       0.92      0.91      0.92       109
weighted avg       0.93      0.93      0.93       109


✅ XGBoost model saved


#### Cell 6 — XGBoost Findings
Gradient boosting — handles non-linear patterns and feature interactions.
Best Accuracy (0.927) and F1 (0.886) so far.
ROC-AUC ties with LR at 0.965 — both equally strong at ranking.
PMOS+ Recall = 0.86 — same as RF, misses more cases than LR.
CV-AUC = 0.971 ± 0.016 — very stable across folds.
LR still leads clinically due to higher recall on minority class.

### Cell 7 — SVM

In [7]:
pipe_svm = Pipeline([
    ('scaler', StandardScaler()),
    ('model', SVC(
        kernel='rbf',
        probability=True,
        random_state=42,
        C=1.0
    ))
])

pipe_svm.fit(X_train, y_train)

svm_results = evaluate_model(
    'SVM (RBF Kernel)',
    pipe_svm, X_train, y_train,
    X_test, y_test
)
results.append(svm_results)

# Save model
with open('artifacts/model_svm.pkl', 'wb') as f:
    pickle.dump(pipe_svm, f)
print('\n✅ SVM model saved')


  SVM (RBF Kernel)
  Accuracy       : 0.8899
  F1 Score       : 0.8378
  ROC-AUC        : 0.9587
  PR-AUC         : 0.9423
  CV ROC-AUC     : 0.9636 ± 0.0179

  Classification Report:
              precision    recall  f1-score   support

       PMOS-       0.93      0.90      0.92        73
       PMOS+       0.82      0.86      0.84        36

    accuracy                           0.89       109
   macro avg       0.87      0.88      0.88       109
weighted avg       0.89      0.89      0.89       109


✅ SVM model saved


####  SVM (RBF Kernel) Findings
- Non-linear kernel — maps features to higher dimensional space.
- Weakest performer across all metrics on this dataset.
- RBF kernel needs larger datasets and careful C/gamma tuning to perform well.
- PMOS+ Recall = 0.86 — same as RF and XGB, misses more cases than LR.
- Will be excluded from final ensemble — adds no value over existing models.